In [1]:
import numpy as np
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,2"
import timeautoencoder as tae
import timediffusion as tdf
import DP as dp
import pandas as pd
import torch
import time
import process_edited as pce
import correl as correl
import Metrics as mt
import matplotlib.pyplot as plt
import random
import predictive_metrics as pdm
import warnings
warnings.simplefilter(action='ignore', category=RuntimeWarning)

In [2]:
# filename = f'../Dataset/Multi-Sequence Data/nasdaq100_2019.csv'
filename = f'../Dataset/hiv.csv.gz'
# Read dataframe
print(filename)
real_df = pd.read_csv(filename).fillna(0).drop_duplicates()

unique_id = real_df['patient_id'].unique()
np.random.seed(123123)
unique_id = np.random.choice(unique_id, size=int(len(unique_id)*0.85), replace=False)
real_df = real_df[real_df.patient_id.isin(unique_id)].reset_index(drop=True)

real_df1 = real_df.drop(['patient_id','date'], axis=1)

has_na = real_df.isna().any().any()
print("Are there any missing values in the DataFrame? ", has_na)
print(f"real_df.shape: {real_df.shape}")

##################################################################################################################
print("Preprocess Data...")
device = 'cuda'

processed_data = torch.load("../Dataset/save/processed_data.pt")
time_info = torch.load("../Dataset/save/time_info.pt")

_, _real_data = pce.convert_to_table(real_df1, processed_data, threshold = 1)

../Dataset/hiv.csv.gz


/tmp/ipykernel_41061/1664926261.py:5: DtypeWarning: Columns (171) have mixed types. Specify dtype option on import or set low_memory=False.
  real_df = pd.read_csv(filename).fillna(0).drop_duplicates()


Are there any missing values in the DataFrame?  False
real_df.shape: (2118656, 172)
Preprocess Data...


100%|██████████| 170/170 [00:00<00:00, 175.41it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]


In [3]:
# Dataset list:  traffic, pollution, hurricane, AirQuality, fraud, nasdaq
# Load the tensor
# real_df = torch.load(f'Data/traffic_real.pt')
_synth_data = torch.load(f'../results/timeautodiff/syn_timeautodiff_test_pad.pt')

In [4]:
synth_data, _synth_data = pce.convert_to_table(real_df1, _synth_data, 1)

100%|██████████| 170/170 [00:00<00:00, 176.59it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]


In [5]:
_real_data = _real_data[:40000,:,:]
_synth_data = _synth_data[:40000,:,:]

In [ ]:
#################### Evaluate Metrics #################### 
iterations = 2000
result_disc = []; result_pred = []; result_tmp = []; result_corr = []

for i in range(10):
    random_integers = [random.randint(0, len(real_df)-1) for _ in range(2000)]
    
    torch.cuda.empty_cache()
    a = mt.discriminative_score_metrics(_real_data, _synth_data, iterations)
    b = pdm.predictive_score_metrics(_real_data, _synth_data, 5)
    c = mt.temp_disc_score(_real_data, _synth_data, iterations)
    d = correl.final_correlation(real_df.drop(['patient_id','date'], axis=1).iloc[random_integers,:], 
                                 synth_data.iloc[random_integers,:])
    
    result_disc.append(a)
    result_pred.append(b)
    result_tmp.append(c)
    result_corr.append(d)

In [14]:
print(f"Discriminative score: {np.mean(result_disc)}(sd {np.std(result_disc)})")
print(f"Predictive score (syn pred real's MAE): {np.mean(result_pred):2f}(sd {np.std(result_pred):2f})")
print(f"Temperal Discriminative score: {np.mean(result_tmp):2f}(sd {np.std(result_tmp):2f})")
print(f"Columns-wise corr distance:  {np.mean(result_corr):2f}（sd {np.std(result_corr):2f}）")

Discriminative score: 0.5(sd 0.0)
Predictive score (syn pred real's MAE): 0.082192(sd 0.022791)
Temperal Discriminative score: 0.499806(sd 0.000283)
Columns-wise corr distance:  35.096867（sd 46.250607）
